# Quick SLM — 12 · Fixed-base control — fine-tune SmolLM2 on the identical corpus

Section 10.4 of the paper finds that the grounding failure is arithmetic rather
than state-blindness: the model reads the state, copies identifiers out of it,
and gets the derived quantity wrong. That points at the base model, whose
arithmetic score is the weakest column of its capability battery.

This notebook runs the control. It fine-tunes an **existing** base model of
comparable size but stronger arithmetic on the **identical corpus**, with the
identical split and the identical evaluation, and reports the same judge-free
grounding metric.

| outcome | reading |
|---|---|
| it grounds | the negative result is about **this base checkpoint**; the corpus is learnable and pretraining is what failed |
| it does not ground | the result is about the **corpus or the scale** — the more interesting outcome, and it redirects the successor away from the optimizer |

**The corpus must be re-tokenized.** SmolLM2 has its own vocabulary, so the
packed `.bin` files from notebook 04b are unusable here — their token ids mean
different things. This notebook rebuilds the corpus from the raw shards, adds the
14 special tokens to SmolLM2's tokenizer, resizes the embedding matrix, and packs
with that tokenizer into a **separate** directory. Nothing v1 owns is touched.

The train/validation split is derived from the examples with the same seed, so
the same 44 counterfactual pairs are held out and the numbers are comparable to
Table 15 line for line.

## 1. Mount and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors tokenizers datasketch tqdm bitsandbytes


## 2. Locate the repository

In [ ]:
import sys, json, math, random
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/quick-slm/code')
framework_dir = REPO_DIR / 'framework'
assert framework_dir.is_dir(), f'{framework_dir} not found'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from v1.quick_slm_trainer.support import require_framework
require_framework('v1', REPO_DIR)
from v1.quick_slm_trainer.paths import Layout
from v1.quick_slm_trainer.config import sft_v1

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
layout = Layout(drive_root=DRIVE_ROOT)
cfg = sft_v1()

# Everything this notebook writes goes here. v1's artefacts are read-only.
CONTROL_DIR = DRIVE_ROOT / 'control_smollm2'
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
print('framework OK; writing to', CONTROL_DIR)

## 3. Rebuild the corpus and the held-out split

Identical to notebooks 07 and 08: load the raw shards, validate, deduplicate,
and split with the same seed. The split operates on examples and never touches a
tokenizer, so the held-out set here is the same set of examples the 103M model
was evaluated on.

In [ ]:
from v1.quick_slm_trainer.sft import load_and_validate
from v1.quick_slm_trainer.sft.corpus import category_histogram, paired_integrity
from v1.quick_slm_trainer.sft.dedup import dedup_examples
from v1.quick_slm_trainer.sft.pack import split_examples

validated, val_stats, _ = load_and_validate(layout, cfg.sft)
deduped = dedup_examples(validated, threshold=cfg.sft.dedup_jaccard,
                         num_perm=cfg.sft.minhash_perms, progress=True)
corpus = list(deduped)            # 04b ran with REBALANCE = False
train_ex, val_ex = split_examples(corpus, val_fraction=cfg.sft.val_fraction)

# The counts must match the pack that trained the 103M model, or this is not the
# same corpus and the comparison is void.
recorded = json.loads(layout.sft_stats_path.read_text())
want_train = recorded.get('pack', {}).get('train', {}).get('examples_in')
want_val   = recorded.get('pack', {}).get('val', {}).get('examples_in')
assert (want_train, want_val) == (len(train_ex), len(val_ex)), (
    f'rebuilt {len(train_ex)}/{len(val_ex)} but v1 packed {want_train}/{want_val}; '
    'the corpus differs and the control would not be comparable')

n_pairs, broken = paired_integrity(val_ex)
print(f'train {len(train_ex):,}   val {len(val_ex):,}   (matches v1)')
print(f'held-out counterfactual pairs: {n_pairs}   broken: {len(broken)}')
print('val by category:', category_histogram(val_ex))

## 4. Load the base model and extend its tokenizer

SmolLM2-135M is the closest reference: this project's width and vocabulary size
were taken from the SmolLM family, and it is trained on a multi-trillion-token
budget against this project's 10B, which is exactly the difference the control is
meant to isolate.

The 14 special tokens are added as real tokens and the embedding matrix is
resized. Without this the envelope would be tokenized as ordinary text and the
model would be learning a different, harder task than the 103M model did.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from v1.quick_slm_trainer.tokenizer import SPECIAL_TOKENS

BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'      # or 'EleutherAI/pythia-160m'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
before = len(tok)
already = [t for t in SPECIAL_TOKENS if t in tok.get_vocab()]
added = tok.add_special_tokens({'additional_special_tokens': list(SPECIAL_TOKENS)})
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print(f'{BASE_MODEL}: vocab {before:,} -> {len(tok):,}  (+{added} new)')
if already:
    print(f'  already present, reused: {already}')

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype)
n_new = len(tok) - before
model.resize_token_embeddings(len(tok))

# resize_token_embeddings initialises the new rows from the model's init
# distribution, which for a pretrained checkpoint is noise relative to the
# trained rows. The envelope tokens are exactly the ones the model must EMIT, so
# random output rows mean it cannot produce <think> or </response> and the whole
# format collapses -- a first run of this notebook reached 6.7% well-formed
# against v1's 99.9% and generated repetition loops.
#
# The standard remedy is to seed each new row from the mean of the trained rows,
# which starts them at the centre of the learned space rather than outside it.
# v1 needed none of this: its special tokens were in the vocabulary from the
# first pretraining step (Section 5).
if n_new > 0:
    with torch.no_grad():
        emb = model.get_input_embeddings().weight
        mean_in = emb[:before].mean(dim=0, keepdim=True)
        emb[before:] = mean_in.repeat(n_new, 1)
        out = model.get_output_embeddings()
        if out is not None and out.weight is not emb:      # untied head
            mean_out = out.weight[:before].mean(dim=0, keepdim=True)
            out.weight[before:] = mean_out.repeat(n_new, 1)
    tied = model.get_output_embeddings() is None or model.get_output_embeddings().weight is emb
    print(f'  seeded {n_new} new embedding rows from the mean of the trained rows'
          f'  (output head {"tied" if tied else "separate, also seeded"})')

model = model.to(device)
print(f'  {sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {device}')

# Sanity: the envelope must round-trip through this tokenizer as single tokens.
for t in ('<think>', '</think>', '<response>', '</response>', '<|im_start|>', '<|im_end|>'):
    ids = tok(t, add_special_tokens=False)['input_ids']
    assert len(ids) == 1, f'{t!r} tokenizes to {len(ids)} tokens, not 1 -- the envelope will not be learnable'
print('  envelope tokens all map to single ids')

# The packing context cannot exceed what the model can attend over.
MODEL_MAX = getattr(model.config, 'max_position_embeddings', cfg.data.ctx)
CTX = min(cfg.data.ctx, MODEL_MAX)
if CTX != cfg.data.ctx:
    print(f'  NOTE: model context is {MODEL_MAX}; packing at {CTX} instead of {cfg.data.ctx}.')
    print('        Examples longer than this are dropped, so the corpus is not identical.')
else:
    print(f'  context {CTX} matches the v1 pack')

## 5. Re-tokenize and pack

The same `pack_split` the v1 pipeline uses, given this tokenizer and a private
layout so nothing under `sft/` is overwritten.

In [ ]:
import dataclasses
from v1.quick_slm_trainer.sft.pack import pack_split

# A Layout rooted at the control directory: same structure, separate files.
control_layout = Layout(drive_root=CONTROL_DIR, local_root=Path('/content/control_local'))
control_layout.mkdirs_sft()

sft_cfg = dataclasses.replace(cfg.sft, ctx=CTX)

train_stats = pack_split(control_layout, 'train', train_ex, tok, sft_cfg)
val_stats   = pack_split(control_layout, 'val',   val_ex,   tok, sft_cfg)
print()
print(f'train windows {train_stats.windows:,}  tokens {train_stats.total_tokens:,}'
      f'  scored {train_stats.scored_fraction:.1%}')
print(f'val   windows {val_stats.windows:,}  tokens {val_stats.total_tokens:,}')
print(f'dropped too long: train {train_stats.dropped_too_long}, val {val_stats.dropped_too_long}')
if train_stats.dropped_too_long:
    print('  WARNING: examples were dropped; note this when comparing to v1.')

## 6. Train

The v1 fine-tuning schedule, from the frozen config: peak 5e-5 into a cosine to
5e-6, 200 warmup steps, 12 sequences per optimizer step, three epochs. The step
count follows from this corpus under this tokenizer, so it will differ slightly
from v1's 900; the schedule shape is what is held fixed.

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from v1.quick_slm_trainer.sft.dataset import MaskedWindowDataset
from v1.quick_slm_trainer.schedule import cosine_with_warmup

EPOCHS = 3
IDS   = control_layout.sft_bin('train', 'ids')
MASK  = control_layout.sft_bin('train', 'mask')
V_IDS = control_layout.sft_bin('val', 'ids')
V_MASK= control_layout.sft_bin('val', 'mask')

micro, accum = cfg.run.micro_batch, cfg.run.grad_accum
tokens_per_step = micro * accum * CTX
total_steps = (train_stats.total_tokens * EPOCHS) // tokens_per_step
assert total_steps > cfg.optim.warmup_steps, (
    f'{total_steps} steps is not longer than the {cfg.optim.warmup_steps}-step warmup; '
    'lower grad_accum (this is the v1 defect of Section 9.5)')
print(f'{total_steps} optimizer steps, {tokens_per_step:,} tokens each, warmup {cfg.optim.warmup_steps}')

ds = MaskedWindowDataset(IDS, MASK, CTX, start_window=0, seed=1337)
# MaskedWindowDataset yields a plain (input_ids, labels) tuple, so a collated
# batch is a two-element list, not a dict. Unpacking it as a mapping is the
# error this line exists to prevent.
dl = DataLoader(ds, batch_size=micro, num_workers=2, pin_memory=(device == 'cuda'))

_probe = next(iter(dl))
assert isinstance(_probe, (list, tuple)) and len(_probe) == 2, (
    f'expected a 2-tuple from the loader, got {type(_probe).__name__}')
print(f'batch shapes: ids {tuple(_probe[0].shape)}  labels {tuple(_probe[1].shape)}')
del _probe

opt = torch.optim.AdamW(model.parameters(), lr=cfg.optim.lr_peak,
                        betas=tuple(cfg.optim.betas), weight_decay=cfg.optim.weight_decay)
model.gradient_checkpointing_enable()
model.config.use_cache = False       # incompatible with checkpointing; silences the warning
model.train()

step = 0
it = iter(dl)
pbar = tqdm(total=total_steps, desc='train')
while step < total_steps:
    opt.zero_grad(set_to_none=True)
    losses = []
    for _ in range(accum):
        try:
            ids_b, lab_b = next(it)
        except StopIteration:
            it = iter(dl)
            ids_b, lab_b = next(it)
        ids_b = ids_b.to(device, non_blocking=True)
        lab_b = lab_b.to(device, non_blocking=True)
        out = model(input_ids=ids_b, labels=lab_b)
        (out.loss / accum).backward()
        losses.append(out.loss.item())
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.optim.grad_clip)
    lr = cosine_with_warmup(step, lr_peak=cfg.optim.lr_peak, lr_min=cfg.optim.lr_min,
                            warmup_steps=cfg.optim.warmup_steps, total_steps=total_steps)
    for g in opt.param_groups:
        g['lr'] = lr
    opt.step()
    step += 1
    pbar.update(1)
    if step % 25 == 0:
        pbar.set_postfix(loss=f'{sum(losses)/len(losses):.3f}', lr=f'{lr:.2e}')
pbar.close()

model.config.use_cache = True        # generation in section 7 wants it back
OUT = CONTROL_DIR / 'sft_final'
model.save_pretrained(str(OUT)); tok.save_pretrained(str(OUT))
print('saved', OUT)

## 7. Generate on the held-out split

The same protocol as notebook 07: greedy, special tokens preserved on decode,
truncated at the first closed response.

In [ ]:
from v1.quick_slm_trainer.template import render_prompt

BATCH, MAX_NEW = 16, 192
model.eval()
tok.padding_side = 'left'

prompts = [render_prompt(e) for e in val_ex]
responses = []
for i in tqdm(range(0, len(prompts), BATCH), desc='generate'):
    chunk = prompts[i:i+BATCH]
    enc = tok(chunk, return_tensors='pt', padding=True, truncation=True,
              max_length=CTX - MAX_NEW).to(device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    cut = enc['input_ids'].shape[1]
    for g in gen:
        text = tok.decode(g[cut:], skip_special_tokens=False)
        for t in (tok.pad_token or '', tok.eos_token or ''):
            if t:
                text = text.replace(t, '')
        end = text.find('</response>')
        responses.append(text if end < 0 else text[:end + len('</response>')])

def expected_of(ex):
    for turn in ex.turns:
        calls = getattr(turn, 'calls', None)
        if calls is not None:
            return json.dumps(list(calls), ensure_ascii=False)
    return '[]'

record = {'label': 'control-' + BASE_MODEL.split('/')[-1], 'hf_dir': str(OUT),
          'outputs': [{'prompt': e.turns[0].text, 'expected': expected_of(e),
                       'response': r, 'category': e.category,
                       'subtype': e.meta.get('subtype',''), 'spec_id': e.meta.get('spec_id'),
                       'group': e.meta.get('group'), 'state': e.state}
                      for e, r in zip(val_ex, responses)]}
(CONTROL_DIR / 'control_outputs.json').write_text(json.dumps(record, indent=2, default=str))
print('wrote', CONTROL_DIR / 'control_outputs.json')
print('\n--- first response ---\n' + responses[0][:400])

## 8. Score — the judge-free metrics from Sections 10.3 and 10.4

No judge is needed to answer the question. Exact match against the oracle gives
the grounded-pair rate directly, and the same failure-mode split as Table 16.
The v1 numbers are shown alongside.

In [ ]:
import collections
FREE = {'answer': {'text'}}

def first_call(t):
    i = t.find('[')
    if i < 0: return None
    d = 0; ins = False; esc = False
    for j in range(i, len(t)):
        c = t[j]
        if ins:
            esc = (c == '\\') and not esc
            if c == '"' and not esc: ins = False
            continue
        if c == '"': ins = True
        elif c == '[': d += 1
        elif c == ']':
            d -= 1
            if d == 0:
                try: v = json.loads(t[i:j+1])
                except Exception: return None
                return v[0] if isinstance(v, list) and v and isinstance(v[0], dict) and 'name' in v[0] else None
    return None

def canon(c):
    if not c: return None
    n = c.get('name'); f = FREE.get(n, set())
    return (n, tuple(sorted((k, str(v).strip()) for k, v in (c.get('arguments') or {}).items() if k not in f)))

def wilson(k, n, z=1.96):
    if not n: return (0.0, 0.0)
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0, c-h), min(1, c+h))

outs = record['outputs']

# --- structural / semantic, comparable to Table 14 ---
wf = ok_tool = ok_full = n_parse = 0
for x in outs:
    g, e = first_call(x['response']), first_call(x['expected'])
    if g is None: continue
    wf += 1
    if e is None: continue
    n_parse += 1
    if g.get('name') == e.get('name'): ok_tool += 1
    if canon(g) == canon(e): ok_full += 1
N = len(outs)
print(f'well-formed call      {wf}/{N} = {wf/N:.1%}      (v1 99.9%)')
print(f'right tool            {ok_tool}/{n_parse} = {ok_tool/n_parse:.1%}   (v1 77.8%)')
print(f'right tool + args     {ok_full}/{n_parse} = {ok_full/n_parse:.1%}   (v1 60.9%)')

# --- grounded pairs, comparable to Table 15 ---
pairs = collections.defaultdict(list)
for x in outs:
    if x['category'] == 'state_memory_conflict' and x.get('group'):
        pairs[x['group']].append(x)
comp = {g: v for g, v in pairs.items() if len(v) == 2}
both = one = 0
identical = 0
for g, v in comp.items():
    ga, gb = canon(first_call(v[0]['response'])), canon(first_call(v[1]['response']))
    ca = ga == canon(first_call(v[0]['expected']))
    cb = gb == canon(first_call(v[1]['expected']))
    if ga == gb: identical += 1
    if ca and cb: both += 1
    elif ca or cb: one += 1
lo, hi = wilson(both, len(comp))
print(f'\n=== THE ANSWER ===')
print(f'grounded pairs (both branches exact): {both}/{len(comp)} = {both/len(comp):.1%}'
      f'  CI [{lo:.1%}, {hi:.1%}]')
print(f'   v1 for comparison:                 4/44 = 9.1%  CI [3.6%, 21.2%]')
print(f'exactly one branch: {one}   identical call on both: {identical} ({identical/len(comp):.1%}, v1 27.3%)')

# --- failure modes, comparable to Table 16 ---
modes = collections.Counter()
for g, v in comp.items():
    for x in v:
        gc, ec = first_call(x['response']), first_call(x['expected'])
        if not gc or not ec: modes['unparsed'] += 1; continue
        ga, ea = gc.get('arguments') or {}, ec.get('arguments') or {}
        if gc.get('name') != ec.get('name'): modes['wrong tool'] += 1; continue
        wrong = [k for k in ea if str(ga.get(k,'')).strip() != str(ea[k]).strip()]
        if not wrong and set(ga) <= set(ea): modes['correct'] += 1
        elif wrong and all(isinstance(ea[k], (int, float)) and isinstance(ga.get(k), (int, float)) for k in wrong):
            modes['right tool, wrong number'] += 1
        else: modes['right tool, other arg wrong'] += 1
tot = sum(modes.values())
print(f'\nfailure modes over {tot} branches   (v1: correct 21.6%, wrong number 35.2%, wrong tool 38.6%)')
for k, v in modes.most_common():
    print(f'  {k:<30}{v:>4}  {v/tot:.1%}')

## 9. Score the same responses with both judges

The judge-free numbers above already answer the question. This section adds the
two language-model judges so that every cell of the comparison in Section 10 is
filled by the same three scorings, and so the control is directly comparable to
the paper's Tables 13 to 16 rather than only to its exact-match columns.

Both judges are loaded in 4-bit, one at a time, with the previous model freed in
between; three models do not fit at once. Each is smoke-tested on three cases
before the full pass, because a judge whose replies do not parse produces 1,386
scores of &minus;1 and an hour of wasted time &mdash; which is exactly what the
first run of notebook 10 did with a reasoning model and a 64-token budget.

Set `RUN_JUDGES = False` to skip this entirely; Sections 8 and 10 still work from
exact match alone.

In [ ]:
RUN_JUDGES = True

# (name, disable-thinking) -- Qwen3 is a reasoning model and needs the flag, or a
# generation budget large enough to hold a thinking block, or both.
JUDGES = [
    ('google/gemma-4-31B-it-qat-q4_0-unquantized', False),
    ('Qwen/Qwen3.6-27B', True),
]
JUDGE_MAX_NEW = 512

judge_scores = {}     # judge name -> list of scores aligned to record['outputs']

if RUN_JUDGES:
    import gc, re
    from transformers import BitsAndBytesConfig

    # The trained model is no longer needed; free it before loading a 27B judge.
    try:
        del model
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    JUDGE_RUBRIC = """You are evaluating one output from a small (103M parameter) tool-calling model that has just been fine-tuned.

The model is given a user request and a set of tools. A correct response reasons briefly, then emits the tool call shown under EXPECTED -- the same tool, with the same argument values. Judge the substance of the call, not its punctuation or whitespace.

Score the response from 0 to 5:
- 0: empty, repetition, or no tool call at all
- 1: a tool call, but unrelated to the request
- 2: plausible tool, wrong task; or the right tool with an argument the user never supplied
- 3: right tool, wrong argument value
- 4: right tool and right arguments, with a flaw in the reasoning or the formatting
- 5: the expected call, with reasoning that supports it

If a STATE block appears in the prompt it is authoritative and outranks any MEMORY block. A response that follows memory against a conflicting state is wrong, however fluent it reads.

Be strict. This is a 103M model and most responses should score 0-2.

PROMPT:
{prompt}

EXPECTED (the call a correct response makes):
{expected}

MODEL RESPONSE:
{response}

Respond with EXACTLY this format and nothing else:
SCORE: <integer 0-5>
REASON: <one short sentence>"""

    _S_RE = re.compile(r'SCORE:\s*([0-5])\b', re.IGNORECASE)

    def _parse_reply(reply):
        # Drop any thinking block and take the LAST committed score: a reasoning
        # model quotes the rubric's numbers while deliberating.
        body = reply.rsplit('</think>', 1)[-1]
        hits = _S_RE.findall(body) or _S_RE.findall(reply)
        return int(hits[-1]) if hits else -1

    _q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)

    for jname, no_think in JUDGES:
        print(f'\n=== judge: {jname} ===')
        jt = AutoTokenizer.from_pretrained(jname)
        jm = AutoModelForCausalLM.from_pretrained(jname, quantization_config=_q4,
                                                  device_map='auto').eval()

        def _tmpl(msg):
            kw = dict(tokenize=False, add_generation_prompt=True)
            if no_think:
                try:
                    return jt.apply_chat_template([{'role': 'user', 'content': msg}],
                                                  enable_thinking=False, **kw)
                except TypeError:
                    pass
            return jt.apply_chat_template([{'role': 'user', 'content': msg}], **kw)

        @torch.no_grad()
        def _score(o):
            msg = JUDGE_RUBRIC.format(prompt=o['prompt'][:800], expected=o['expected'],
                                      response=o['response'][:800])
            enc = jt(_tmpl(msg), return_tensors='pt').to(jm.device)
            g = jm.generate(**enc, max_new_tokens=JUDGE_MAX_NEW, do_sample=False,
                            pad_token_id=jt.eos_token_id)
            return _parse_reply(jt.decode(g[0][enc['input_ids'].shape[1]:],
                                          skip_special_tokens=True))

        probe = [_score(o) for o in record['outputs'][:3]]
        print('  smoke test:', probe)
        assert any(p >= 0 for p in probe), (
            f'{jname} produced no parseable score on any probe. Raise '
            'JUDGE_MAX_NEW or check the thinking flag before the full pass.')

        scores = []
        for i, o in enumerate(tqdm(record['outputs'], desc=f'  {jname.split("/")[-1]}', leave=False)):
            scores.append(_score(o))
            if i == 49 and all(s < 0 for s in scores):
                raise RuntimeError(f'{jname}: first 50 replies all unparsed -- stopping.')
        judge_scores[jname] = scores
        bad = sum(1 for s in scores if s < 0)
        good = [s for s in scores if s >= 0]
        print(f'  unparsed {bad}/{len(scores)}   mean {sum(good)/max(1,len(good)):.3f}')

        # Persist after each judge so a later failure does not lose this one.
        (CONTROL_DIR / 'control_judge_scores.json').write_text(
            json.dumps(judge_scores, indent=2))

        del jm, jt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print('\nwrote', CONTROL_DIR / 'control_judge_scores.json')
else:
    print('RUN_JUDGES is False -- exact-match scoring only.')

## 10. The full comparison

Three models by three scorings. The 103M rows are read from the files notebooks
07 and 10 wrote; the control row is what this notebook just produced. Exact match
is computed identically for all three from the saved generations, so that column
is the one to trust across rows &mdash; it depends on no judge and no rubric.

Read down the exact-match grounded column. That is the control.

In [ ]:
LOGS_DIR = DRIVE_ROOT / 'logs'

def load_outputs(path):
    path = Path(path)
    return json.loads(path.read_text())['outputs'] if path.exists() else None

def exact_metrics(outs):
    """Judge-free: well-formed, right tool, right tool+args, grounded pairs."""
    wf = ok_tool = ok_full = n = 0
    for x in outs:
        g, e = first_call(x['response']), first_call(x['expected'])
        if g is None:
            continue
        wf += 1
        if e is None:
            continue
        n += 1
        if g.get('name') == e.get('name'):
            ok_tool += 1
        if canon(g) == canon(e):
            ok_full += 1
    pairs = collections.defaultdict(list)
    for x in outs:
        if x['category'] == 'state_memory_conflict' and x.get('group'):
            pairs[x['group']].append(x)
    comp = {k: v for k, v in pairs.items() if len(v) == 2}
    both = ident = 0
    for v in comp.values():
        ga = canon(first_call(v[0]['response']))
        gb = canon(first_call(v[1]['response']))
        if ga == gb:
            ident += 1
        if ga == canon(first_call(v[0]['expected'])) and gb == canon(first_call(v[1]['expected'])):
            both += 1
    return {'n': len(outs), 'well_formed': wf / len(outs),
            'right_tool': ok_tool / max(1, n), 'right_tool_args': ok_full / max(1, n),
            'grounded': both, 'pairs': len(comp), 'identical_pairs': ident}

def judged_metrics(scores, outs):
    """Mean score and grounded pairs (both branches >= 4) from a judge's scores."""
    ok = [s for s in scores if s >= 0]
    pairs = collections.defaultdict(list)
    for s, x in zip(scores, outs):
        if x['category'] == 'state_memory_conflict' and x.get('group') and s >= 0:
            pairs[x['group']].append(s)
    comp = {k: v for k, v in pairs.items() if len(v) == 2}
    return {'mean': sum(ok) / max(1, len(ok)), 'unparsed': len(scores) - len(ok),
            'ge4': sum(1 for s in ok if s >= 4) / max(1, len(ok)),
            'grounded': sum(1 for v in comp.values() if min(v) >= 4), 'pairs': len(comp)}

HDR = '{:<24}{:<14}{:>7}{:>8}{:>8}{:>11}{:>11}'
ROW = '{:<24}{:<14}{:>7}{:>8}{:>8}{:>11}{:>11}{}'

def pct(x):
    return '{:.1%}'.format(x)

rows = []

for name, probe_p, gemma_p, qwen_p in [
    ('103M base', LOGS_DIR / 'sft_probe_outputs_base.json',
     LOGS_DIR / 'sft_scored_outputs_base.json',
     LOGS_DIR / 'sft_scored_outputs_base_second.json'),
    ('103M sft-final', LOGS_DIR / 'sft_probe_outputs_sft-final.json',
     LOGS_DIR / 'sft_scored_outputs_sft-final.json',
     LOGS_DIR / 'sft_scored_outputs_sft-final_second.json'),
]:
    outs = load_outputs(probe_p)
    if outs is None:
        rows.append((name, '(no cached generations)', '', '', '', '', '', ''))
        continue
    em = exact_metrics(outs)
    rows.append((name, 'exact match', '-', '-', pct(em['right_tool']),
                 pct(em['right_tool_args']),
                 '{}/{}'.format(em['grounded'], em['pairs']), ''))
    for lbl, p in (('Gemma 4', gemma_p), ('Qwen 3.6', qwen_p)):
        sc = load_outputs(p)
        if sc is None:
            rows.append((name, lbl, '(not run)', '', '', '', '', ''))
            continue
        jm = judged_metrics([x['score'] for x in sc], sc)
        note = '  [{} unparsed]'.format(jm['unparsed']) if jm['unparsed'] else ''
        rows.append((name, lbl, '{:.2f}'.format(jm['mean']), pct(jm['ge4']), '-', '-',
                     '{}/{}'.format(jm['grounded'], jm['pairs']), note))

# --- the control row ---
outs = record['outputs']
em = exact_metrics(outs)
ctl = ('control ' + BASE_MODEL.split('/')[-1])[:23]
rows.append((ctl, 'exact match', '-', '-', pct(em['right_tool']),
             pct(em['right_tool_args']),
             '{}/{}'.format(em['grounded'], em['pairs']), ''))
for jname, scores in judge_scores.items():
    jm = judged_metrics(scores, outs)
    note = '  [{} unparsed]'.format(jm['unparsed']) if jm['unparsed'] else ''
    short = 'Gemma 4' if 'gemma' in jname.lower() else jname.split('/')[-1]
    rows.append((ctl, short, '{:.2f}'.format(jm['mean']), pct(jm['ge4']), '-', '-',
                 '{}/{}'.format(jm['grounded'], jm['pairs']), note))

print(HDR.format('model', 'scoring', 'mean', 'ge4', 'tool', 'tool+args', 'grounded'))
print('-' * 84)
last = None
for r in rows:
    if last is not None and r[0] != last:
        print('-' * 84)
    last = r[0]
    print(ROW.format(*r))

print('\nThe exact-match grounded column is the control: it depends on no judge.')
print('v1 reference from the paper: 4/44 grounded, 77.8% tool, 60.9% tool+args.')

## 11. What the result means

Read the well-formed rate before the grounded rate. A control that never learnt
the envelope has not tested the corpus, and its 0 % grounded says nothing about
grounding.

| well-formed | grounded | reading |
|---|---|---|
| near v1's 99.9 % | materially above 9.1 % | the corpus is learnable and the v1 base is what failed; Section 10.4's arithmetic diagnosis is confirmed |
| near v1's 99.9 % | near 9.1 % | the failure is the corpus or the scale, not this base checkpoint; the swap specifications that define grounding as a subtraction are the first thing to change |
| far below 99.9 % | anything | **the run is broken, not a null.** The model never learnt the format, so nothing downstream is a measurement |

The first run of this notebook fell in the third row: 6.7 % well-formed, 0/44
grounded, and repetition loops in the generations, because the 14 added special
tokens carried untrained embeddings. Section 4 now seeds them from the mean of
the trained rows and asserts each maps to a single token id.

Send `control_outputs.json` and the printed summary either way.

In [ ]:
summary = {
    'base_model': BASE_MODEL, 'ctx': CTX, 'steps': total_steps,
    'well_formed': wf/N, 'right_tool': ok_tool/n_parse, 'right_tool_args': ok_full/n_parse,
    'grounded_pairs': both, 'pairs': len(comp), 'grounded_rate': both/len(comp),
    'identical_call_pairs': identical, 'exactly_one': one,
    'failure_modes': dict(modes),
    'v1_reference': {'grounded': '4/44 = 9.1%', 'right_tool': 0.778,
                     'right_tool_args': 0.609, 'identical_pairs': 12},
}
(CONTROL_DIR / 'control_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))